# Genie Space Optimization — Salama Insurance

This notebook improves the **Insurance Operations and Analytics** Genie Space (`01f13962554916408eede5637ffecb13`) by:

1. **Table Descriptions** — Adding `COMMENT ON TABLE` for all 19 tables
2. **Column Descriptions** — Adding `COMMENT` on every key column across core tables
3. **Foreign Key Constraints** — Defining PK/FK relationships for correct joins
4. **Pre-Joined Views** — Creating denormalized views for faster Genie responses
5. **Knowledge Store Guidance** — SQL Expressions, Example Queries, Text Instructions, and Benchmark Questions

> **Impact**: Column/table descriptions are the #1 factor for Genie accuracy. FK constraints ensure correct joins. SQL expressions define business KPIs.

In [0]:
%sql
-- ============================================
-- CELL 2: Table-Level Descriptions
-- ============================================

COMMENT ON TABLE salama_insurance.salama_silver.fact_fraud_investigation IS 'Fact table containing fraud investigation records. Each row is a unique investigation linked to a claim. Tracks fraud scores (0-100), investigation costs, fraud amounts detected, recovery amounts, investigation duration in days, investigator assignments, status (INITIATED/IN_PROGRESS/COMPLETED/CLOSED), and findings (FRAUD_CONFIRMED/FRAUD_SUSPECTED/INCONCLUSIVE/NO_FRAUD).';

COMMENT ON TABLE salama_insurance.salama_silver.fact_claim IS 'Fact table for insurance claims. Contains claim amounts (claimed, approved, paid, reserve), days to report and settle, ratios (claim, approval, payment), business line, claim type, status, risk rating and score, aging buckets, and adjuster assignments.';

COMMENT ON TABLE salama_insurance.salama_silver.dim_customer IS 'Customer dimension table with demographics including customer type (Individual/Corporate), name, nationality, emirate, city, risk rating, customer segment, and active status. Use IS_CURRENT=true for latest records.';

COMMENT ON TABLE salama_insurance.salama_silver.dim_policy IS 'Policy dimension table with policy details including product code, business line, premium amount, sum insured, policy status, sales channel, and agent code.';

COMMENT ON TABLE salama_insurance.salama_silver.dim_claim IS 'Claim dimension with detailed claim attributes including claim type, status, and classification details.';

COMMENT ON TABLE salama_insurance.salama_silver.dim_date IS 'Date dimension for time-based analysis. Join using date key columns from fact tables (e.g., INVESTIGATION_DATE_KEY, INCIDENT_DATE_KEY).';

COMMENT ON TABLE salama_insurance.salama_silver.dim_agent IS 'Agent dimension with agent details, hierarchy, and performance information.';

COMMENT ON TABLE salama_insurance.salama_silver.dim_product IS 'Product dimension with product codes, names, and business line classifications.';

COMMENT ON TABLE salama_insurance.salama_silver.dim_account IS 'Account dimension for financial and operational accounts.';

COMMENT ON TABLE salama_insurance.salama_silver.dim_employee IS 'Employee dimension with department, position, salary, nationality, and organizational hierarchy.';

COMMENT ON TABLE salama_insurance.salama_silver.fact_claim_payment IS 'Claim payment transactions fact table tracking individual payment disbursements against claims.';

COMMENT ON TABLE salama_insurance.salama_silver.fact_financial IS 'Financial transactions fact table for accounting and financial reporting.';

COMMENT ON TABLE salama_insurance.salama_silver.fact_investment IS 'Investment portfolio fact table tracking investment positions and returns.';

COMMENT ON TABLE salama_insurance.salama_silver.fact_litigation IS 'Litigation cases fact table tracking legal proceedings related to claims.';

COMMENT ON TABLE salama_insurance.salama_silver.fact_policy IS 'Policy lifecycle fact table tracking policy issuance, renewals, and cancellations.';

COMMENT ON TABLE salama_insurance.salama_silver.fact_premium IS 'Premium collection fact table tracking premium payments and collections.';

COMMENT ON TABLE salama_insurance.salama_silver.fact_sales_funnel IS 'Sales pipeline and conversion fact table tracking leads through the sales process.';

COMMENT ON TABLE salama_insurance.salama_silver.fraud_ai_results IS 'AI-generated insights, priority levels, and recommendations per investigation. Links to fact_fraud_investigation via INVESTIGATION_ID.';

COMMENT ON TABLE salama_insurance.salama_silver.fraud_investigation IS 'Alternate fraud investigation table with similar structure to fact_fraud_investigation. Contains investigation details, scores, and outcomes.';

In [0]:
%sql
-- ============================================
-- CELL 3: Column Descriptions - fact_fraud_investigation
-- ============================================

ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN FACT_FRAUD_KEY COMMENT 'Surrogate key for the fraud investigation fact record';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN INVESTIGATION_ID COMMENT 'Unique business identifier for the fraud investigation (e.g., INV-001)';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN CLAIM_KEY COMMENT 'Foreign key linking to fact_claim.CLAIM_KEY';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN INVESTIGATION_DATE_KEY COMMENT 'Foreign key to dim_date for investigation start date';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN COMPLETION_DATE_KEY COMMENT 'Foreign key to dim_date for investigation completion date';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN FRAUD_SCORE COMMENT 'Fraud likelihood score from 0 (no fraud) to 100 (certain fraud). Scores above 75 are considered high risk.';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN INVESTIGATION_COST COMMENT 'Total cost of the investigation in USD';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN FRAUD_AMOUNT_DETECTED COMMENT 'Total fraud amount discovered during investigation in USD';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN RECOVERY_AMOUNT COMMENT 'Amount successfully recovered from the fraud in USD';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN INVESTIGATION_DAYS COMMENT 'Number of days the investigation took from start to completion';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN FRAUD_DETECTION_RATE COMMENT 'Rate of fraud detection as a decimal (0.0 to 1.0)';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN INVESTIGATOR_ID COMMENT 'ID of the investigator assigned to this case';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN INVESTIGATION_STATUS COMMENT 'Current status: INITIATED (new), IN_PROGRESS (active), COMPLETED (done), CLOSED (archived)';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN FINDINGS COMMENT 'Investigation outcome: FRAUD_CONFIRMED (proven fraud), FRAUD_SUSPECTED (likely fraud), INCONCLUSIVE (unclear), NO_FRAUD (clean)';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN CREATED_AT COMMENT 'Timestamp when the investigation record was created';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN UPDATED_AT COMMENT 'Timestamp of the last update to this record';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN FR_DATE COMMENT 'The primary date of the fraud investigation, use this for time-based analysis';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN MONTHYEAR COMMENT 'Human-readable month-year label (e.g., Jan-2025) for display purposes';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN MONTHYEAR_SORT COMMENT 'Numeric sort key for MONTHYEAR to ensure chronological ordering';
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN FRAUD_KEY_NEW COMMENT 'Alternate key for linking to related fraud records';

In [0]:
%sql
-- ============================================
-- CELL 4: Column Descriptions - fact_claim
-- ============================================

ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN FACT_CLAIM_KEY COMMENT 'Surrogate key for the claim fact record';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CLAIM_ID COMMENT 'Unique business identifier for the claim';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN POLICY_ID COMMENT 'Foreign key to dim_policy.POLICY_ID';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CUSTOMER_KEY COMMENT 'Foreign key to dim_customer.CUSTOMER_KEY';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN POLICY_KEY COMMENT 'Foreign key to dim_policy.POLICY_KEY';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CLAIM_KEY COMMENT 'Claim surrogate key, used to join with fact_fraud_investigation.CLAIM_KEY';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN INCIDENT_DATE_KEY COMMENT 'Foreign key to dim_date for when the incident occurred';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN REPORTED_DATE_KEY COMMENT 'Foreign key to dim_date for when the claim was reported';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CREATED_DATE_KEY COMMENT 'Foreign key to dim_date for claim creation date';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN SETTLEMENT_DATE_KEY COMMENT 'Foreign key to dim_date for settlement date';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CL_DATE COMMENT 'Primary claim date timestamp, use for time-based claim analysis';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CLAIMED_AMOUNT COMMENT 'Total amount claimed by the policyholder in USD';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN APPROVED_AMOUNT COMMENT 'Amount approved by the insurer in USD';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN PAID_AMOUNT COMMENT 'Amount actually paid out in USD';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN RESERVE_AMOUNT COMMENT 'Reserved amount set aside for the claim in USD';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN DAYS_TO_REPORT COMMENT 'Number of days between the incident date and when the claim was reported';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN DAYS_TO_SETTLE COMMENT 'Number of days from claim creation to settlement';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CLAIM_RATIO COMMENT 'Ratio of claimed amount to sum insured';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN APPROVAL_RATIO COMMENT 'Ratio of approved amount to claimed amount (approved/claimed)';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN PAYMENT_RATIO COMMENT 'Ratio of paid amount to approved amount (paid/approved)';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN BUSINESS_LINE COMMENT 'Insurance business line (e.g., Motor, Medical, Property, Life)';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CLAIM_TYPE COMMENT 'Type of claim filed';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CLAIM_STATUS COMMENT 'Current claim processing status';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN ADJUSTER_ID COMMENT 'ID of the claims adjuster assigned';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN RISK_RATING COMMENT 'Risk category assigned to this claim';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN RISK_SCORE COMMENT 'Numeric risk score for the claim';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CLAIM_NUMBER COMMENT 'Human-readable claim number';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CLAIM_AGING_BUCKET COMMENT 'Aging category for the claim (e.g., 0-30 days, 31-60 days)';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN MONTHYEAR COMMENT 'Human-readable month-year label for display';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN MONTHYEAR_SORT COMMENT 'Numeric sort key for chronological ordering';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN FRAUD_KEY_NEW COMMENT 'Key for linking to fraud investigation records';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN INVESTIGATION_STATUS COMMENT 'Investigation status from linked fraud investigation';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN INVESTIGATION_DAYS COMMENT 'Investigation duration from linked fraud investigation';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN RECOVERY_AMOUNT COMMENT 'Recovery amount from linked fraud investigation in USD';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN FINDINGS COMMENT 'Investigation findings from linked fraud investigation';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN FACT_FRAUD_KEY COMMENT 'Key linking to the fraud investigation fact record';
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN INVESTIGATIONDAY_BUCKET COMMENT 'Bucket category for investigation duration (e.g., 0-7 days, 8-14 days)';

In [0]:
%sql
-- ============================================
-- CELL 5: Column Descriptions - dim_customer
-- ============================================

ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN CUSTOMER_KEY COMMENT 'Surrogate key for customer dimension, join to fact_claim.CUSTOMER_KEY';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN CUSTOMER_ID COMMENT 'Unique business identifier for the customer';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN CUSTOMER_TYPE COMMENT 'Customer classification: Individual or Corporate';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN CUSTOMER_NAME COMMENT 'Full name of the customer';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN REGISTRATION_NUMBER COMMENT 'Business registration number (for corporate customers)';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN DATE_OF_BIRTH COMMENT 'Customer date of birth (for individual customers)';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN ESTABLISHMENT_DATE COMMENT 'Company establishment date (for corporate customers)';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN NATIONALITY COMMENT 'Customer nationality';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN EMIRATES COMMENT 'UAE emirate where the customer is located (e.g., Dubai, Abu Dhabi, Sharjah)';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN CITY COMMENT 'City of residence';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN EMAIL COMMENT 'Customer email address';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN PHONE_NUMBER COMMENT 'Customer phone number';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN RISK_RATING COMMENT 'Customer risk rating category';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN CUSTOMER_SEGMENT COMMENT 'Customer segment classification (e.g., Premium, Standard, VIP)';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN ONBOARDING_DATE COMMENT 'Date when customer was first onboarded';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN LAST_ACTIVITY_DATE COMMENT 'Date of customers last activity or interaction';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN IS_ACTIVE COMMENT 'Whether the customer account is currently active (true/false)';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN EFFECTIVE_DATE COMMENT 'SCD2 effective date for this customer record version';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN EXPIRY_DATE COMMENT 'SCD2 expiry date for this customer record version';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN IS_CURRENT COMMENT 'SCD2 flag: true means this is the latest version of the customer record';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN ROW_VERSION COMMENT 'Version number for tracking changes to this customer record';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN SOURCE_SYSTEM COMMENT 'Source system that originated this customer record';
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN RISK_SCORE COMMENT 'Numeric risk score for the customer';

In [0]:
%sql
-- ============================================
-- CELL 6: Column Descriptions - dim_policy
-- ============================================

ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_KEY COMMENT 'Surrogate key for policy dimension';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_ID COMMENT 'Unique business identifier for the policy, join to fact_claim.POLICY_ID';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_NUMBER COMMENT 'Human-readable policy number';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN CUSTOMER_ID COMMENT 'Customer who owns this policy';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN PRODUCT_CODE COMMENT 'Product code identifying the insurance product type';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN BUSINESS_LINE COMMENT 'Insurance business line (e.g., Motor, Medical, Property, Life)';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_HIERARCHY_LEVEL1 COMMENT 'Top-level policy classification hierarchy';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_HIERARCHY_LEVEL2 COMMENT 'Mid-level policy classification hierarchy';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_HIERARCHY_LEVEL3 COMMENT 'Detailed policy classification hierarchy';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_START_DATE COMMENT 'Policy coverage start date';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_END_DATE COMMENT 'Policy coverage end date';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN PREMIUM_AMOUNT COMMENT 'Annual premium amount in USD';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN SUM_INSURED COMMENT 'Total sum insured (coverage limit) in USD';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_STATUS COMMENT 'Current policy status (Active, Expired, Cancelled, etc.)';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN SALES_CHANNEL COMMENT 'Distribution channel through which the policy was sold';
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN AGENT_CODE COMMENT 'Code of the agent who sold this policy';

In [0]:
%sql
-- ============================================
-- CELL 7: Column Descriptions - fraud_ai_results
-- ============================================

ALTER TABLE salama_insurance.salama_silver.fraud_ai_results ALTER COLUMN INVESTIGATION_ID COMMENT 'Links to fact_fraud_investigation.INVESTIGATION_ID - the investigation this AI analysis belongs to';
ALTER TABLE salama_insurance.salama_silver.fraud_ai_results ALTER COLUMN AI_INSIGHTS COMMENT 'AI-generated analytical insights about the investigation patterns and anomalies';
ALTER TABLE salama_insurance.salama_silver.fraud_ai_results ALTER COLUMN AI_PRIORITY COMMENT 'AI-assigned priority level for the investigation (e.g., Critical, High, Medium, Low)';
ALTER TABLE salama_insurance.salama_silver.fraud_ai_results ALTER COLUMN AI_RECOMMENDATIONS COMMENT 'AI-recommended next actions for the investigation team';
ALTER TABLE salama_insurance.salama_silver.fraud_ai_results ALTER COLUMN RUN_TIMESTAMP COMMENT 'Timestamp when the AI analysis was executed';

In [0]:
%sql
-- ============================================
-- CELL 8: Foreign Key Constraints
-- ============================================
-- Using business/join keys as PKs (these are what Genie uses for joins)
-- All constraints are NOT ENFORCED (informational for Genie and query optimizer)

-- Step 0: Drop any existing constraints from prior runs
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation DROP CONSTRAINT IF EXISTS fk_fraud_claim;
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation DROP CONSTRAINT IF EXISTS fk_ai_investigation;
ALTER TABLE salama_insurance.salama_silver.fact_claim DROP CONSTRAINT IF EXISTS fk_claim_customer;
ALTER TABLE salama_insurance.salama_silver.fact_claim DROP CONSTRAINT IF EXISTS fk_claim_policy;
ALTER TABLE salama_insurance.salama_silver.fraud_ai_results DROP CONSTRAINT IF EXISTS fk_ai_investigation;

ALTER TABLE salama_insurance.salama_silver.dim_customer DROP CONSTRAINT IF EXISTS pk_dim_customer;
ALTER TABLE salama_insurance.salama_silver.dim_policy DROP CONSTRAINT IF EXISTS pk_dim_policy;
ALTER TABLE salama_insurance.salama_silver.fact_claim DROP CONSTRAINT IF EXISTS pk_fact_claim;
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation DROP CONSTRAINT IF EXISTS pk_fact_fraud;
ALTER TABLE salama_insurance.salama_silver.fraud_ai_results DROP CONSTRAINT IF EXISTS pk_fraud_ai;

-- Step 1: Set join key columns NOT NULL (required for PK constraints)
ALTER TABLE salama_insurance.salama_silver.dim_customer ALTER COLUMN CUSTOMER_KEY SET NOT NULL;
ALTER TABLE salama_insurance.salama_silver.dim_policy ALTER COLUMN POLICY_KEY SET NOT NULL;
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CLAIM_KEY SET NOT NULL;
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN CUSTOMER_KEY SET NOT NULL;
ALTER TABLE salama_insurance.salama_silver.fact_claim ALTER COLUMN POLICY_KEY SET NOT NULL;
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN INVESTIGATION_ID SET NOT NULL;
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ALTER COLUMN CLAIM_KEY SET NOT NULL;
ALTER TABLE salama_insurance.salama_silver.fraud_ai_results ALTER COLUMN INVESTIGATION_ID SET NOT NULL;

-- Step 2: Primary Keys on join target columns
ALTER TABLE salama_insurance.salama_silver.dim_customer ADD CONSTRAINT pk_dim_customer PRIMARY KEY (CUSTOMER_KEY) NOT ENFORCED;
ALTER TABLE salama_insurance.salama_silver.dim_policy ADD CONSTRAINT pk_dim_policy PRIMARY KEY (POLICY_KEY) NOT ENFORCED;
ALTER TABLE salama_insurance.salama_silver.fact_claim ADD CONSTRAINT pk_fact_claim PRIMARY KEY (CLAIM_KEY) NOT ENFORCED;
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ADD CONSTRAINT pk_fact_fraud PRIMARY KEY (INVESTIGATION_ID) NOT ENFORCED;
ALTER TABLE salama_insurance.salama_silver.fraud_ai_results ADD CONSTRAINT pk_fraud_ai PRIMARY KEY (INVESTIGATION_ID) NOT ENFORCED;

-- Step 3: Foreign Keys
ALTER TABLE salama_insurance.salama_silver.fact_fraud_investigation ADD CONSTRAINT fk_fraud_claim FOREIGN KEY (CLAIM_KEY) REFERENCES salama_insurance.salama_silver.fact_claim(CLAIM_KEY) NOT ENFORCED;
ALTER TABLE salama_insurance.salama_silver.fact_claim ADD CONSTRAINT fk_claim_customer FOREIGN KEY (CUSTOMER_KEY) REFERENCES salama_insurance.salama_silver.dim_customer(CUSTOMER_KEY) NOT ENFORCED;
ALTER TABLE salama_insurance.salama_silver.fact_claim ADD CONSTRAINT fk_claim_policy FOREIGN KEY (POLICY_KEY) REFERENCES salama_insurance.salama_silver.dim_policy(POLICY_KEY) NOT ENFORCED;
ALTER TABLE salama_insurance.salama_silver.fraud_ai_results ADD CONSTRAINT fk_ai_investigation FOREIGN KEY (INVESTIGATION_ID) REFERENCES salama_insurance.salama_silver.fact_fraud_investigation(INVESTIGATION_ID) NOT ENFORCED;

In [0]:
%sql
-- ============================================
-- CELL 9: Pre-Joined Analysis Views
-- ============================================

-- View 1: Comprehensive Fraud Investigation Detail
-- Joins investigations + claims + customers + policies + AI results
CREATE OR REPLACE VIEW salama_insurance.salama_silver.v_fraud_investigation_detail AS
SELECT
  -- Investigation fields
  fi.INVESTIGATION_ID,
  fi.FRAUD_SCORE,
  fi.INVESTIGATION_COST,
  fi.FRAUD_AMOUNT_DETECTED,
  fi.RECOVERY_AMOUNT,
  fi.INVESTIGATION_DAYS,
  fi.FRAUD_DETECTION_RATE,
  fi.INVESTIGATOR_ID,
  fi.INVESTIGATION_STATUS,
  fi.FINDINGS,
  fi.FR_DATE AS investigation_date,
  fi.MONTHYEAR AS investigation_month,
  fi.MONTHYEAR_SORT,
  -- Calculated metrics
  ROUND(fi.RECOVERY_AMOUNT / NULLIF(fi.FRAUD_AMOUNT_DETECTED, 0) * 100, 1) AS recovery_rate_pct,
  ROUND((fi.RECOVERY_AMOUNT - fi.INVESTIGATION_COST) / NULLIF(fi.INVESTIGATION_COST, 0) * 100, 1) AS investigation_roi_pct,
  fi.RECOVERY_AMOUNT - fi.INVESTIGATION_COST AS net_recovery_benefit,
  CASE WHEN fi.FRAUD_SCORE > 75 THEN 'High' WHEN fi.FRAUD_SCORE > 50 THEN 'Medium' ELSE 'Low' END AS risk_tier,
  -- Claim fields
  fc.CLAIM_ID,
  fc.CLAIMED_AMOUNT,
  fc.APPROVED_AMOUNT,
  fc.PAID_AMOUNT,
  fc.RESERVE_AMOUNT,
  fc.DAYS_TO_REPORT,
  fc.DAYS_TO_SETTLE,
  fc.BUSINESS_LINE,
  fc.CLAIM_TYPE,
  fc.CLAIM_STATUS,
  fc.CLAIM_AGING_BUCKET,
  fc.RISK_RATING AS claim_risk_rating,
  fc.RISK_SCORE AS claim_risk_score,
  -- Customer fields
  dc.CUSTOMER_ID,
  dc.CUSTOMER_NAME,
  dc.CUSTOMER_TYPE,
  dc.NATIONALITY,
  dc.EMIRATES,
  dc.CITY,
  dc.RISK_RATING AS customer_risk_rating,
  dc.CUSTOMER_SEGMENT,
  -- Policy fields
  dp.POLICY_ID,
  dp.POLICY_NUMBER,
  dp.PRODUCT_CODE,
  dp.BUSINESS_LINE AS policy_business_line,
  dp.PREMIUM_AMOUNT,
  dp.SUM_INSURED,
  dp.POLICY_STATUS,
  dp.SALES_CHANNEL,
  -- AI results
  ai.AI_INSIGHTS,
  ai.AI_PRIORITY,
  ai.AI_RECOMMENDATIONS
FROM salama_insurance.salama_silver.fact_fraud_investigation fi
LEFT JOIN salama_insurance.salama_silver.fact_claim fc ON fi.CLAIM_KEY = fc.CLAIM_KEY
LEFT JOIN salama_insurance.salama_silver.dim_customer dc ON fc.CUSTOMER_KEY = dc.CUSTOMER_KEY AND dc.IS_CURRENT = true
LEFT JOIN salama_insurance.salama_silver.dim_policy dp ON fc.POLICY_KEY = dp.POLICY_KEY
LEFT JOIN salama_insurance.salama_silver.fraud_ai_results ai ON fi.INVESTIGATION_ID = ai.INVESTIGATION_ID;

COMMENT ON TABLE salama_insurance.salama_silver.v_fraud_investigation_detail IS 'Pre-joined view combining fraud investigations with claims, customers, policies, and AI results. Use this for comprehensive fraud analysis without writing complex joins. Includes calculated fields: recovery_rate_pct, investigation_roi_pct, net_recovery_benefit, risk_tier.';

-- View 2: Claims Analysis
-- Joins claims + customers + policies for claims-focused analysis
CREATE OR REPLACE VIEW salama_insurance.salama_silver.v_claims_analysis AS
SELECT
  fc.CLAIM_ID,
  fc.CLAIM_NUMBER,
  fc.CLAIMED_AMOUNT,
  fc.APPROVED_AMOUNT,
  fc.PAID_AMOUNT,
  fc.RESERVE_AMOUNT,
  fc.DAYS_TO_REPORT,
  fc.DAYS_TO_SETTLE,
  fc.CLAIM_RATIO,
  fc.APPROVAL_RATIO,
  fc.PAYMENT_RATIO,
  fc.BUSINESS_LINE,
  fc.CLAIM_TYPE,
  fc.CLAIM_STATUS,
  fc.CLAIM_AGING_BUCKET,
  fc.RISK_RATING,
  fc.RISK_SCORE,
  fc.CL_DATE AS claim_date,
  fc.MONTHYEAR AS claim_month,
  fc.MONTHYEAR_SORT,
  ROUND(fc.APPROVED_AMOUNT / NULLIF(fc.CLAIMED_AMOUNT, 0) * 100, 1) AS approval_rate_pct,
  ROUND(fc.PAID_AMOUNT / NULLIF(fc.APPROVED_AMOUNT, 0) * 100, 1) AS payment_rate_pct,
  -- Customer fields
  dc.CUSTOMER_ID,
  dc.CUSTOMER_NAME,
  dc.CUSTOMER_TYPE,
  dc.NATIONALITY,
  dc.EMIRATES,
  dc.CITY,
  dc.CUSTOMER_SEGMENT,
  dc.RISK_RATING AS customer_risk_rating,
  -- Policy fields
  dp.POLICY_ID,
  dp.POLICY_NUMBER,
  dp.PRODUCT_CODE,
  dp.BUSINESS_LINE AS policy_business_line,
  dp.PREMIUM_AMOUNT,
  dp.SUM_INSURED,
  dp.POLICY_STATUS,
  dp.SALES_CHANNEL
FROM salama_insurance.salama_silver.fact_claim fc
LEFT JOIN salama_insurance.salama_silver.dim_customer dc ON fc.CUSTOMER_KEY = dc.CUSTOMER_KEY AND dc.IS_CURRENT = true
LEFT JOIN salama_insurance.salama_silver.dim_policy dp ON fc.POLICY_KEY = dp.POLICY_KEY;

COMMENT ON TABLE salama_insurance.salama_silver.v_claims_analysis IS 'Pre-joined view combining claims with customer and policy details. Use for claims analysis, approval rates, settlement patterns, and business line performance without writing complex joins.';

## Cell 10: SQL Expressions to Add in the Genie Space Knowledge Store

> **Instructions**: Go to the Genie Space settings → Knowledge Store → SQL Expressions and add each of the following. These teach Genie reusable business metrics.

---

### Measures

| Name | Expression | Synonyms |
| --- | --- | --- |
| Recovery Rate | `ROUND(SUM(RECOVERY_AMOUNT) / NULLIF(SUM(FRAUD_AMOUNT_DETECTED), 0) * 100, 1)` | recovery percentage, amount recovered rate, recovery ratio |
| Fraud ROI | `ROUND((SUM(RECOVERY_AMOUNT) - SUM(INVESTIGATION_COST)) / NULLIF(SUM(INVESTIGATION_COST), 0) * 100, 1)` | return on investigation, investigation ROI, fraud return |
| Claim Approval Rate | `ROUND(SUM(APPROVED_AMOUNT) / NULLIF(SUM(CLAIMED_AMOUNT), 0) * 100, 1)` | approval percentage, approval ratio, claim approval |
| Average Fraud Score | `ROUND(AVG(FRAUD_SCORE), 1)` | mean fraud score, avg score, average risk score |
| Net Recovery Benefit | `SUM(RECOVERY_AMOUNT) - SUM(INVESTIGATION_COST)` | net benefit, profit from investigations, net gain |
| Average Investigation Duration | `ROUND(AVG(INVESTIGATION_DAYS), 1)` | avg duration, mean investigation time, days to investigate |
| Total Fraud Detected | `SUM(FRAUD_AMOUNT_DETECTED)` | total fraud, fraud amount, fraud detected |
| Total Recovery | `SUM(RECOVERY_AMOUNT)` | total recovered, amount recovered, recovery total |

---

### Filters

| Name | Expression | Synonyms |
| --- | --- | --- |
| High Risk Investigation | `FRAUD_SCORE > 75` | high risk, risky, dangerous cases, high fraud score |
| Medium Risk Investigation | `FRAUD_SCORE BETWEEN 50 AND 75` | medium risk, moderate risk |
| Low Risk Investigation | `FRAUD_SCORE < 50` | low risk, safe cases |
| Confirmed Fraud | `FINDINGS = 'FRAUD_CONFIRMED'` | proven fraud, confirmed fraud cases, verified fraud |
| Active Investigation | `INVESTIGATION_STATUS IN ('INITIATED', 'IN_PROGRESS')` | open cases, ongoing, active, current investigations |
| Completed Investigation | `INVESTIGATION_STATUS = 'COMPLETED'` | finished, done investigations, completed cases |
| Large Claims | `CLAIMED_AMOUNT > 100000` | big claims, expensive claims, major claims, high value claims |
| Quick Investigations | `INVESTIGATION_DAYS < 14` | fast investigations, quick turnaround, under 2 weeks |
| Long Investigations | `INVESTIGATION_DAYS > 60` | slow investigations, lengthy, overdue |

---

### Dimensions

| Name | Expression | Synonyms |
| --- | --- | --- |
| Investigation Month | `DATE_TRUNC('month', FR_DATE)` | month, monthly, by month, time period |
| Risk Tier | `CASE WHEN FRAUD_SCORE > 75 THEN 'High' WHEN FRAUD_SCORE > 50 THEN 'Medium' ELSE 'Low' END` | risk level, risk category, risk band, risk group |

In [0]:
import requests, json, uuid

DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

SPACE_ID = "01f13962554916408eede5637ffecb13"
BASE_URL = f"https://{DATABRICKS_HOST}/api/2.0/genie/spaces"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# ── Step 1: GET current space config ──
resp = requests.get(f"{BASE_URL}/{SPACE_ID}?include_serialized_space=true", headers=headers)
resp.raise_for_status()
space_data = resp.json()
config = json.loads(space_data["serialized_space"])
print(f"Space: {space_data['title']} | Current tables: {len(config.get('data_sources', {}).get('tables', []))}")

def new_id():
    return uuid.uuid4().hex

# Table names for qualified column references (required by Genie UI validation)
FI = "fact_fraud_investigation"
FC = "fact_claim"
DC = "dim_customer"
DP = "dim_policy"
S  = "salama_insurance.salama_silver"

# ── Step 2: Add 2 pre-joined views ──
existing = [t["identifier"] for t in config.get("data_sources", {}).get("tables", [])]
views = [
    {"identifier": f"{S}.v_fraud_investigation_detail",
     "description": ["Pre-joined view: investigations + claims + customers + policies + AI results. Includes recovery_rate_pct, investigation_roi_pct, net_recovery_benefit, risk_tier."]},
    {"identifier": f"{S}.v_claims_analysis",
     "description": ["Pre-joined view: claims + customers + policies. Includes approval_rate_pct and payment_rate_pct."]}
]
for v in views:
    if v["identifier"] not in existing:
        config["data_sources"]["tables"].append(v)
config["data_sources"]["tables"].sort(key=lambda t: t["identifier"])
print(f"Tables: {len(config['data_sources']['tables'])}")

# ── Step 3: SQL Expressions - All columns are TABLE-QUALIFIED ──
if "instructions" not in config:
    config["instructions"] = {}

config["instructions"]["sql_snippets"] = {
    "measures": sorted([
        {"id": new_id(), "display_name": "Recovery Rate",
         "sql": [f"ROUND(SUM({FI}.RECOVERY_AMOUNT) / NULLIF(SUM({FI}.FRAUD_AMOUNT_DETECTED), 0) * 100, 1)"],
         "synonyms": ["recovery percentage", "amount recovered rate", "recovery ratio"]},
        {"id": new_id(), "display_name": "Fraud ROI",
         "sql": [f"ROUND((SUM({FI}.RECOVERY_AMOUNT) - SUM({FI}.INVESTIGATION_COST)) / NULLIF(SUM({FI}.INVESTIGATION_COST), 0) * 100, 1)"],
         "synonyms": ["return on investigation", "investigation ROI", "fraud return"]},
        {"id": new_id(), "display_name": "Claim Approval Rate",
         "sql": [f"ROUND(SUM({FC}.APPROVED_AMOUNT) / NULLIF(SUM({FC}.CLAIMED_AMOUNT), 0) * 100, 1)"],
         "synonyms": ["approval percentage", "approval ratio", "claim approval"]},
        {"id": new_id(), "display_name": "Average Fraud Score",
         "sql": [f"ROUND(AVG({FI}.FRAUD_SCORE), 1)"],
         "synonyms": ["mean fraud score", "avg score", "average risk score"]},
        {"id": new_id(), "display_name": "Net Recovery Benefit",
         "sql": [f"SUM({FI}.RECOVERY_AMOUNT) - SUM({FI}.INVESTIGATION_COST)"],
         "synonyms": ["net benefit", "profit from investigations", "net gain"]},
        {"id": new_id(), "display_name": "Average Investigation Duration",
         "sql": [f"ROUND(AVG({FI}.INVESTIGATION_DAYS), 1)"],
         "synonyms": ["avg duration", "mean investigation time", "days to investigate"]},
        {"id": new_id(), "display_name": "Total Fraud Detected",
         "sql": [f"SUM({FI}.FRAUD_AMOUNT_DETECTED)"],
         "synonyms": ["total fraud", "fraud amount", "fraud detected"]},
        {"id": new_id(), "display_name": "Total Recovery",
         "sql": [f"SUM({FI}.RECOVERY_AMOUNT)"],
         "synonyms": ["total recovered", "amount recovered", "recovery total"]},
    ], key=lambda x: x["id"]),
    "filters": sorted([
        {"id": new_id(), "display_name": "High Risk Investigation",
         "sql": [f"{FI}.FRAUD_SCORE > 75"],
         "synonyms": ["high risk", "risky", "dangerous cases", "high fraud score"]},
        {"id": new_id(), "display_name": "Medium Risk Investigation",
         "sql": [f"{FI}.FRAUD_SCORE BETWEEN 50 AND 75"],
         "synonyms": ["medium risk", "moderate risk"]},
        {"id": new_id(), "display_name": "Low Risk Investigation",
         "sql": [f"{FI}.FRAUD_SCORE < 50"],
         "synonyms": ["low risk", "safe cases"]},
        {"id": new_id(), "display_name": "Confirmed Fraud",
         "sql": [f"{FI}.FINDINGS = 'FRAUD_CONFIRMED'"],
         "synonyms": ["proven fraud", "confirmed fraud cases", "verified fraud"]},
        {"id": new_id(), "display_name": "Active Investigation",
         "sql": [f"{FI}.INVESTIGATION_STATUS IN ('INITIATED', 'IN_PROGRESS')"],
         "synonyms": ["open cases", "ongoing", "active", "current investigations"]},
        {"id": new_id(), "display_name": "Completed Investigation",
         "sql": [f"{FI}.INVESTIGATION_STATUS = 'COMPLETED'"],
         "synonyms": ["finished", "done investigations", "completed cases"]},
        {"id": new_id(), "display_name": "Large Claims",
         "sql": [f"{FC}.CLAIMED_AMOUNT > 100000"],
         "synonyms": ["big claims", "expensive claims", "major claims"]},
        {"id": new_id(), "display_name": "Quick Investigations",
         "sql": [f"{FI}.INVESTIGATION_DAYS < 14"],
         "synonyms": ["fast investigations", "quick turnaround", "under 2 weeks"]},
        {"id": new_id(), "display_name": "Long Investigations",
         "sql": [f"{FI}.INVESTIGATION_DAYS > 60"],
         "synonyms": ["slow investigations", "lengthy", "overdue"]},
    ], key=lambda x: x["id"])
}
print("Added 8 measures + 9 filters (all table-qualified)")

# ── Step 4: Example SQL Queries (all table-qualified, sorted by id) ──
config["instructions"]["example_question_sqls"] = sorted([
    {"id": new_id(), "question": ["What is the total fraud detected?"],
     "sql": [f"SELECT SUM(fi.FRAUD_AMOUNT_DETECTED) AS total_fraud_detected, SUM(fi.RECOVERY_AMOUNT) AS total_recovered, COUNT(*) AS total_investigations, ROUND(SUM(fi.RECOVERY_AMOUNT) / NULLIF(SUM(fi.FRAUD_AMOUNT_DETECTED), 0) * 100, 1) AS recovery_rate_pct FROM {S}.{FI} fi"]},
    {"id": new_id(), "question": ["Show fraud by business line"],
     "sql": [f"SELECT fc.BUSINESS_LINE, COUNT(fi.INVESTIGATION_ID) AS investigations, SUM(fi.FRAUD_AMOUNT_DETECTED) AS fraud_detected, SUM(fi.RECOVERY_AMOUNT) AS recovered, ROUND(SUM(fi.RECOVERY_AMOUNT) / NULLIF(SUM(fi.FRAUD_AMOUNT_DETECTED), 0) * 100, 1) AS recovery_rate_pct FROM {S}.{FI} fi JOIN {S}.{FC} fc ON fi.CLAIM_KEY = fc.CLAIM_KEY GROUP BY fc.BUSINESS_LINE ORDER BY fraud_detected DESC"]},
    {"id": new_id(), "question": ["Who are the top investigators?"],
     "sql": [f"SELECT fi.INVESTIGATOR_ID, COUNT(*) AS total_cases, ROUND(AVG(fi.FRAUD_SCORE), 1) AS avg_fraud_score, SUM(fi.FRAUD_AMOUNT_DETECTED) AS total_fraud_found, SUM(fi.RECOVERY_AMOUNT) AS total_recovered, ROUND((SUM(fi.RECOVERY_AMOUNT) - SUM(fi.INVESTIGATION_COST)) / NULLIF(SUM(fi.INVESTIGATION_COST), 0) * 100, 1) AS roi_pct FROM {S}.{FI} fi GROUP BY fi.INVESTIGATOR_ID ORDER BY total_recovered DESC"]},
    {"id": new_id(), "question": ["Monthly fraud trends"],
     "sql": [f"SELECT fi.MONTHYEAR, COUNT(*) AS investigations, SUM(fi.FRAUD_AMOUNT_DETECTED) AS fraud_detected, SUM(fi.RECOVERY_AMOUNT) AS recovered, ROUND(AVG(fi.FRAUD_SCORE), 1) AS avg_score FROM {S}.{FI} fi GROUP BY fi.MONTHYEAR, fi.MONTHYEAR_SORT ORDER BY fi.MONTHYEAR_SORT"]},
    {"id": new_id(), "question": ["Show high risk cases"],
     "sql": [f"SELECT fi.INVESTIGATION_ID, fi.FRAUD_SCORE, fi.FRAUD_AMOUNT_DETECTED, fi.RECOVERY_AMOUNT, fi.INVESTIGATION_STATUS, fi.FINDINGS, fi.INVESTIGATOR_ID FROM {S}.{FI} fi WHERE fi.FRAUD_SCORE > 75 ORDER BY fi.FRAUD_SCORE DESC"]},
    {"id": new_id(), "question": ["Fraud analysis by customer"],
     "sql": [f"SELECT dc.CUSTOMER_NAME, dc.CUSTOMER_TYPE, dc.EMIRATES, dc.RISK_RATING AS customer_risk, COUNT(fi.INVESTIGATION_ID) AS investigations, SUM(fi.FRAUD_AMOUNT_DETECTED) AS total_fraud FROM {S}.{FI} fi JOIN {S}.{FC} fc ON fi.CLAIM_KEY = fc.CLAIM_KEY JOIN {S}.{DC} dc ON fc.CUSTOMER_KEY = dc.CUSTOMER_KEY AND dc.IS_CURRENT = true GROUP BY dc.CUSTOMER_NAME, dc.CUSTOMER_TYPE, dc.EMIRATES, dc.RISK_RATING ORDER BY total_fraud DESC LIMIT 20"]},
    {"id": new_id(), "question": ["What is the recovery rate?"],
     "sql": [f"SELECT ROUND(SUM(fi.RECOVERY_AMOUNT) / NULLIF(SUM(fi.FRAUD_AMOUNT_DETECTED), 0) * 100, 1) AS overall_recovery_rate_pct, SUM(fi.RECOVERY_AMOUNT) AS total_recovered, SUM(fi.FRAUD_AMOUNT_DETECTED) AS total_fraud_detected, SUM(fi.INVESTIGATION_COST) AS total_cost, ROUND((SUM(fi.RECOVERY_AMOUNT) - SUM(fi.INVESTIGATION_COST)) / NULLIF(SUM(fi.INVESTIGATION_COST), 0) * 100, 1) AS roi_pct FROM {S}.{FI} fi"]},
    {"id": new_id(), "question": ["Investigation pipeline by status"],
     "sql": [f"SELECT fi.INVESTIGATION_STATUS, COUNT(*) AS count, ROUND(AVG(fi.FRAUD_SCORE), 1) AS avg_score, SUM(fi.FRAUD_AMOUNT_DETECTED) AS fraud_amount, ROUND(AVG(fi.INVESTIGATION_DAYS), 1) AS avg_duration_days FROM {S}.{FI} fi GROUP BY fi.INVESTIGATION_STATUS ORDER BY count DESC"]},
    {"id": new_id(), "question": ["Claim aging analysis"],
     "sql": [f"SELECT fc.CLAIM_AGING_BUCKET, COUNT(*) AS claim_count, SUM(fc.CLAIMED_AMOUNT) AS total_claimed, SUM(fc.APPROVED_AMOUNT) AS total_approved, SUM(fc.PAID_AMOUNT) AS total_paid, ROUND(AVG(fc.DAYS_TO_SETTLE), 1) AS avg_days_to_settle FROM {S}.{FC} fc GROUP BY fc.CLAIM_AGING_BUCKET ORDER BY claim_count DESC"]},
    {"id": new_id(), "question": ["Average investigation duration"],
     "sql": [f"SELECT fi.INVESTIGATION_STATUS, fi.FINDINGS, COUNT(*) AS cases, ROUND(AVG(fi.INVESTIGATION_DAYS), 1) AS avg_days, MIN(fi.INVESTIGATION_DAYS) AS min_days, MAX(fi.INVESTIGATION_DAYS) AS max_days FROM {S}.{FI} fi GROUP BY fi.INVESTIGATION_STATUS, fi.FINDINGS ORDER BY avg_days DESC"]},
    {"id": new_id(), "question": ["Fraud detection by emirate"],
     "sql": [f"SELECT dc.EMIRATES, COUNT(fi.INVESTIGATION_ID) AS investigations, SUM(fi.FRAUD_AMOUNT_DETECTED) AS fraud_detected, SUM(fi.RECOVERY_AMOUNT) AS recovered, ROUND(AVG(fi.FRAUD_SCORE), 1) AS avg_score FROM {S}.{FI} fi JOIN {S}.{FC} fc ON fi.CLAIM_KEY = fc.CLAIM_KEY JOIN {S}.{DC} dc ON fc.CUSTOMER_KEY = dc.CUSTOMER_KEY AND dc.IS_CURRENT = true GROUP BY dc.EMIRATES ORDER BY fraud_detected DESC"]},
    {"id": new_id(), "question": ["Top fraud cases by amount"],
     "sql": [f"SELECT fi.INVESTIGATION_ID, fi.FRAUD_SCORE, fi.FRAUD_AMOUNT_DETECTED, fi.RECOVERY_AMOUNT, fi.FINDINGS, fi.INVESTIGATION_STATUS, fc.BUSINESS_LINE, fc.CLAIMED_AMOUNT FROM {S}.{FI} fi JOIN {S}.{FC} fc ON fi.CLAIM_KEY = fc.CLAIM_KEY ORDER BY fi.FRAUD_AMOUNT_DETECTED DESC LIMIT 20"]},
], key=lambda x: x["id"])
print("Added 12 example SQL queries (all table-qualified)")

# ── Step 5: Text Instructions (single item, multi-content) ──
# Note: Dimensions and Joins defined here since the API doesn't support
# 'dimensions' in sql_snippets or 'join_specs' proto format
config["instructions"]["text_instructions"] = [{
    "id": new_id(),
    "content": [
        f"When users ask about fraud or investigations, query {FI}. For claim details, join to {FC} using CLAIM_KEY. For comprehensive analysis, use v_fraud_investigation_detail view. For claims analysis, use v_claims_analysis view.",
        f"For time-based fraud analysis, use {FI}.FR_DATE. For claim analysis, use {FC}.CL_DATE. {FI}.MONTHYEAR provides human-readable month labels. Always ORDER BY {FI}.MONTHYEAR_SORT for chronological order.",
        f"{FI}.INVESTIGATION_STATUS values: INITIATED, IN_PROGRESS, COMPLETED, CLOSED. {FI}.FINDINGS values: FRAUD_CONFIRMED, FRAUD_SUSPECTED, INCONCLUSIVE, NO_FRAUD.",
        f"All monetary values are in USD. Recovery Rate = SUM({FI}.RECOVERY_AMOUNT) / SUM({FI}.FRAUD_AMOUNT_DETECTED) * 100. ROI = (SUM({FI}.RECOVERY_AMOUNT) - SUM({FI}.INVESTIGATION_COST)) / SUM({FI}.INVESTIGATION_COST) * 100.",
        f"High risk: {FI}.FRAUD_SCORE > 75. Medium risk: 50-75. Low risk: < 50.",
        f"Join relationships: {FI}.CLAIM_KEY = {FC}.CLAIM_KEY. {FC}.CUSTOMER_KEY = {DC}.CUSTOMER_KEY (filter {DC}.IS_CURRENT = true). {FC}.POLICY_KEY = {DP}.POLICY_KEY. fraud_ai_results.INVESTIGATION_ID = {FI}.INVESTIGATION_ID.",
        f"Dimension - Investigation Month: DATE_TRUNC('month', {FI}.FR_DATE). Synonyms: month, monthly, by month.",
        f"Dimension - Risk Tier: CASE WHEN {FI}.FRAUD_SCORE > 75 THEN 'High' WHEN {FI}.FRAUD_SCORE > 50 THEN 'Medium' ELSE 'Low' END. Synonyms: risk level, risk category.",
        f"v_fraud_investigation_detail pre-joins investigations + claims + customers + policies + AI results with recovery_rate_pct, investigation_roi_pct, net_recovery_benefit, risk_tier. v_claims_analysis pre-joins claims + customers + policies with approval_rate_pct, payment_rate_pct."
    ]
}]
print("Added 9 text instructions (includes join paths + dimension definitions)")

# ── Step 6: Remove join_specs if present (API proto format undocumented) ──
config["instructions"].pop("join_specs", None)

# ── Step 7: PATCH the Genie Space ──
table_ids = [t["identifier"] for t in config["data_sources"]["tables"]]
update_payload = {
    "title": space_data["title"],
    "description": space_data.get("description", ""),
    "warehouse_id": space_data["warehouse_id"],
    "table_identifiers": table_ids,
    "serialized_space": json.dumps(config)
}

resp = requests.patch(f"{BASE_URL}/{SPACE_ID}", headers=headers, json=update_payload)
if resp.status_code >= 300:
    print(f"\nPATCH failed ({resp.status_code}): {resp.text[:500]}")
else:
    result = resp.json()
    print(f"\n{'='*55}")
    print(f"GENIE SPACE UPDATED SUCCESSFULLY")
    print(f"{'='*55}")
    print(f"Space: {result.get('title', space_data['title'])}")
    print(f"\nTables ({len(table_ids)}):")
    for t in table_ids:
        m = " <-- NEW VIEW" if "v_fraud" in t or "v_claims" in t else ""
        print(f"  {t}{m}")
    print(f"\nKnowledge Store:")
    print(f"  8 Measures (table-qualified)")
    print(f"  9 Filters  (table-qualified)")
    print(f"  12 Example SQL Queries (table-qualified)")
    print(f"  9 Text Instructions (joins + dimensions + routing)")
    print(f"\nNote: Joins are handled via FK constraints (Cell 8)")
    print(f"  + text instructions. Dimensions are defined in text")
    print(f"  instructions. To add as proper Dimension type in UI:")
    print(f"  Configure > Instructions > SQL Expressions > Add Dimension")

## Cell 11: Example SQL Queries to Add in the Genie Space

> **Instructions**: Go to the Genie Space settings → Knowledge Store → Example SQL Queries and add each query below with its associated question.

---

### 1. Total Fraud Detected
**Question**: "What is the total fraud detected?"
```sql
SELECT
  SUM(FRAUD_AMOUNT_DETECTED) AS total_fraud_detected,
  SUM(RECOVERY_AMOUNT) AS total_recovered,
  COUNT(*) AS total_investigations,
  ROUND(SUM(RECOVERY_AMOUNT) / NULLIF(SUM(FRAUD_AMOUNT_DETECTED), 0) * 100, 1) AS recovery_rate_pct
FROM salama_insurance.salama_silver.fact_fraud_investigation
```

### 2. Fraud by Business Line
**Question**: "Show fraud by business line"
```sql
SELECT
  fc.BUSINESS_LINE,
  COUNT(fi.INVESTIGATION_ID) AS investigations,
  SUM(fi.FRAUD_AMOUNT_DETECTED) AS fraud_detected,
  SUM(fi.RECOVERY_AMOUNT) AS recovered,
  ROUND(SUM(fi.RECOVERY_AMOUNT) / NULLIF(SUM(fi.FRAUD_AMOUNT_DETECTED), 0) * 100, 1) AS recovery_rate_pct
FROM salama_insurance.salama_silver.fact_fraud_investigation fi
JOIN salama_insurance.salama_silver.fact_claim fc ON fi.CLAIM_KEY = fc.CLAIM_KEY
GROUP BY fc.BUSINESS_LINE
ORDER BY fraud_detected DESC
```

### 3. Top Investigators
**Question**: "Who are the top investigators?"
```sql
SELECT
  INVESTIGATOR_ID,
  COUNT(*) AS total_cases,
  ROUND(AVG(FRAUD_SCORE), 1) AS avg_fraud_score,
  SUM(FRAUD_AMOUNT_DETECTED) AS total_fraud_found,
  SUM(RECOVERY_AMOUNT) AS total_recovered,
  ROUND((SUM(RECOVERY_AMOUNT) - SUM(INVESTIGATION_COST)) / NULLIF(SUM(INVESTIGATION_COST), 0) * 100, 1) AS roi_pct
FROM salama_insurance.salama_silver.fact_fraud_investigation
GROUP BY INVESTIGATOR_ID
ORDER BY total_recovered DESC
```

### 4. Monthly Fraud Trends
**Question**: "Monthly fraud trends"
```sql
SELECT
  MONTHYEAR,
  COUNT(*) AS investigations,
  SUM(FRAUD_AMOUNT_DETECTED) AS fraud_detected,
  SUM(RECOVERY_AMOUNT) AS recovered,
  ROUND(AVG(FRAUD_SCORE), 1) AS avg_score
FROM salama_insurance.salama_silver.fact_fraud_investigation
GROUP BY MONTHYEAR, MONTHYEAR_SORT
ORDER BY MONTHYEAR_SORT
```

### 5. High Risk Cases
**Question**: "Show high risk cases"
```sql
SELECT
  INVESTIGATION_ID,
  FRAUD_SCORE,
  FRAUD_AMOUNT_DETECTED,
  RECOVERY_AMOUNT,
  INVESTIGATION_STATUS,
  FINDINGS,
  INVESTIGATOR_ID
FROM salama_insurance.salama_silver.fact_fraud_investigation
WHERE FRAUD_SCORE > 75
ORDER BY FRAUD_SCORE DESC
```

### 6. Customer Fraud Analysis
**Question**: "Fraud analysis by customer"
```sql
SELECT
  dc.CUSTOMER_NAME,
  dc.CUSTOMER_TYPE,
  dc.EMIRATES,
  dc.RISK_RATING AS customer_risk,
  COUNT(fi.INVESTIGATION_ID) AS investigations,
  SUM(fi.FRAUD_AMOUNT_DETECTED) AS total_fraud
FROM salama_insurance.salama_silver.fact_fraud_investigation fi
JOIN salama_insurance.salama_silver.fact_claim fc ON fi.CLAIM_KEY = fc.CLAIM_KEY
JOIN salama_insurance.salama_silver.dim_customer dc ON fc.CUSTOMER_KEY = dc.CUSTOMER_KEY AND dc.IS_CURRENT = true
GROUP BY dc.CUSTOMER_NAME, dc.CUSTOMER_TYPE, dc.EMIRATES, dc.RISK_RATING
ORDER BY total_fraud DESC
LIMIT 20
```

### 7. Recovery Rate Analysis
**Question**: "What is the recovery rate?"
```sql
SELECT
  ROUND(SUM(RECOVERY_AMOUNT) / NULLIF(SUM(FRAUD_AMOUNT_DETECTED), 0) * 100, 1) AS overall_recovery_rate_pct,
  SUM(RECOVERY_AMOUNT) AS total_recovered,
  SUM(FRAUD_AMOUNT_DETECTED) AS total_fraud_detected,
  SUM(INVESTIGATION_COST) AS total_cost,
  ROUND((SUM(RECOVERY_AMOUNT) - SUM(INVESTIGATION_COST)) / NULLIF(SUM(INVESTIGATION_COST), 0) * 100, 1) AS roi_pct
FROM salama_insurance.salama_silver.fact_fraud_investigation
```

### 8. Investigation Pipeline by Status
**Question**: "Investigation pipeline by status"
```sql
SELECT
  INVESTIGATION_STATUS,
  COUNT(*) AS count,
  ROUND(AVG(FRAUD_SCORE), 1) AS avg_score,
  SUM(FRAUD_AMOUNT_DETECTED) AS fraud_amount,
  ROUND(AVG(INVESTIGATION_DAYS), 1) AS avg_duration_days
FROM salama_insurance.salama_silver.fact_fraud_investigation
GROUP BY INVESTIGATION_STATUS
ORDER BY count DESC
```

### 9. Claim Aging Analysis
**Question**: "Claim aging analysis"
```sql
SELECT
  CLAIM_AGING_BUCKET,
  COUNT(*) AS claim_count,
  SUM(CLAIMED_AMOUNT) AS total_claimed,
  SUM(APPROVED_AMOUNT) AS total_approved,
  SUM(PAID_AMOUNT) AS total_paid,
  ROUND(AVG(DAYS_TO_SETTLE), 1) AS avg_days_to_settle
FROM salama_insurance.salama_silver.fact_claim
GROUP BY CLAIM_AGING_BUCKET
ORDER BY claim_count DESC
```

### 10. Average Investigation Duration
**Question**: "Average investigation duration"
```sql
SELECT
  INVESTIGATION_STATUS,
  FINDINGS,
  COUNT(*) AS cases,
  ROUND(AVG(INVESTIGATION_DAYS), 1) AS avg_days,
  MIN(INVESTIGATION_DAYS) AS min_days,
  MAX(INVESTIGATION_DAYS) AS max_days
FROM salama_insurance.salama_silver.fact_fraud_investigation
GROUP BY INVESTIGATION_STATUS, FINDINGS
ORDER BY avg_days DESC
```

### 11. Fraud Detection by Emirate
**Question**: "Fraud detection by emirate"
```sql
SELECT
  dc.EMIRATES,
  COUNT(fi.INVESTIGATION_ID) AS investigations,
  SUM(fi.FRAUD_AMOUNT_DETECTED) AS fraud_detected,
  SUM(fi.RECOVERY_AMOUNT) AS recovered,
  ROUND(AVG(fi.FRAUD_SCORE), 1) AS avg_score
FROM salama_insurance.salama_silver.fact_fraud_investigation fi
JOIN salama_insurance.salama_silver.fact_claim fc ON fi.CLAIM_KEY = fc.CLAIM_KEY
JOIN salama_insurance.salama_silver.dim_customer dc ON fc.CUSTOMER_KEY = dc.CUSTOMER_KEY AND dc.IS_CURRENT = true
GROUP BY dc.EMIRATES
ORDER BY fraud_detected DESC
```

### 12. Top Fraud Cases by Amount
**Question**: "Top fraud cases by amount"
```sql
SELECT
  fi.INVESTIGATION_ID,
  fi.FRAUD_SCORE,
  fi.FRAUD_AMOUNT_DETECTED,
  fi.RECOVERY_AMOUNT,
  fi.FINDINGS,
  fi.INVESTIGATION_STATUS,
  fc.BUSINESS_LINE,
  fc.CLAIMED_AMOUNT
FROM salama_insurance.salama_silver.fact_fraud_investigation fi
JOIN salama_insurance.salama_silver.fact_claim fc ON fi.CLAIM_KEY = fc.CLAIM_KEY
ORDER BY fi.FRAUD_AMOUNT_DETECTED DESC
LIMIT 20
```

## Cell 12: Text Instructions to Add in the Genie Space

> **Instructions**: Go to the Genie Space settings → Knowledge Store → Text Instructions and add the following guidance.

---

### General Query Routing
- When users ask about **fraud** or **investigations**, query the `fact_fraud_investigation` table. For claim details, join to `fact_claim` using `CLAIM_KEY`.
- For comprehensive fraud analysis with customer/policy details, use the pre-joined view `v_fraud_investigation_detail` instead of writing complex joins.
- For claims-focused analysis with customer/policy context, use the `v_claims_analysis` view.

### Date Handling
- For time-based **fraud analysis**, use `FR_DATE` column from `fact_fraud_investigation`.
- For time-based **claim analysis**, use `CL_DATE` column from `fact_claim`.
- `MONTHYEAR` provides human-readable month labels (e.g., "Jan-2025"). **Always ORDER BY `MONTHYEAR_SORT`** for chronological order, not by `MONTHYEAR`.

### Status & Findings Values
- `INVESTIGATION_STATUS` values: **INITIATED** (new case), **IN_PROGRESS** (being investigated), **COMPLETED** (investigation done), **CLOSED** (archived).
- `FINDINGS` values: **FRAUD_CONFIRMED** (proven fraud), **FRAUD_SUSPECTED** (likely but unproven), **INCONCLUSIVE** (insufficient evidence), **NO_FRAUD** (clean case).

### Monetary Calculations
- All monetary values (`FRAUD_AMOUNT_DETECTED`, `RECOVERY_AMOUNT`, `INVESTIGATION_COST`, `CLAIMED_AMOUNT`, `APPROVED_AMOUNT`, `PAID_AMOUNT`) are in **USD**. Round to 2 decimal places.
- **Recovery Rate** = `SUM(RECOVERY_AMOUNT) / SUM(FRAUD_AMOUNT_DETECTED) * 100`
- **ROI / Return on Investigation** = `(SUM(RECOVERY_AMOUNT) - SUM(INVESTIGATION_COST)) / SUM(INVESTIGATION_COST) * 100`
- **Net Benefit** = `SUM(RECOVERY_AMOUNT) - SUM(INVESTIGATION_COST)`

### Risk Classification
- A **high risk** investigation has `FRAUD_SCORE > 75`
- A **medium risk** investigation has `FRAUD_SCORE` between 50 and 75
- A **low risk** investigation has `FRAUD_SCORE < 50`

### Join Relationships
- `fact_fraud_investigation.CLAIM_KEY` → `fact_claim.CLAIM_KEY`
- `fact_claim.CUSTOMER_KEY` → `dim_customer.CUSTOMER_KEY` (filter `IS_CURRENT = true` for latest customer data)
- `fact_claim.POLICY_KEY` → `dim_policy.POLICY_KEY`
- `fraud_ai_results.INVESTIGATION_ID` → `fact_fraud_investigation.INVESTIGATION_ID`

### Pre-Joined Views
- `v_fraud_investigation_detail` pre-joins investigations with claims, customers, policies, and AI results. Includes calculated fields: `recovery_rate_pct`, `investigation_roi_pct`, `net_recovery_benefit`, `risk_tier`.
- `v_claims_analysis` pre-joins claims with customer and policy details. Includes `approval_rate_pct` and `payment_rate_pct`.

## Cell 13: Benchmark Questions for Testing the Genie Space

> **Instructions**: Use these questions to systematically test the Genie Space accuracy. Each question has multiple phrasings. A good Genie Space should handle all phrasings correctly.

---

### Benchmark Set 1: Fraud Totals
- "What is the total fraud detected?"
- "How much fraud was found?"
- "Total fraud amount"
- "Sum of all fraud detected"

**Expected**: Returns `SUM(FRAUD_AMOUNT_DETECTED)` from `fact_fraud_investigation`

---

### Benchmark Set 2: Recovery Rate
- "What is our recovery rate?"
- "How much have we recovered?"
- "Recovery percentage"
- "What percentage of fraud was recovered?"

**Expected**: `SUM(RECOVERY_AMOUNT) / SUM(FRAUD_AMOUNT_DETECTED) * 100`

---

### Benchmark Set 3: Investigator Performance
- "Show top investigators"
- "Best performing investigators"
- "Investigator rankings"
- "Who recovers the most fraud?"

**Expected**: GROUP BY `INVESTIGATOR_ID` with `SUM(RECOVERY_AMOUNT)`, `COUNT(*)`, `AVG(FRAUD_SCORE)`

---

### Benchmark Set 4: Business Line Analysis
- "Fraud by business line"
- "Which lines have most fraud?"
- "Business line fraud breakdown"
- "Compare fraud across business lines"

**Expected**: JOIN to `fact_claim`, GROUP BY `BUSINESS_LINE`

---

### Benchmark Set 5: Time Trends
- "Monthly fraud trends"
- "How has fraud changed over time?"
- "Fraud trend by month"
- "Show me monthly investigation data"

**Expected**: GROUP BY `MONTHYEAR`, ORDER BY `MONTHYEAR_SORT`

---

### Benchmark Set 6: High Risk Analysis
- "Show high risk cases"
- "Which investigations have high fraud scores?"
- "Cases with fraud score above 75"
- "Risky investigations"

**Expected**: `WHERE FRAUD_SCORE > 75`

---

### Benchmark Set 7: Geographic Analysis
- "Fraud by emirate"
- "Which emirates have the most fraud?"
- "Geographic fraud distribution"
- "Fraud detection by region"

**Expected**: JOIN through `fact_claim` to `dim_customer`, GROUP BY `EMIRATES`

---

### Benchmark Set 8: Investigation Pipeline
- "Investigation status breakdown"
- "How many open investigations?"
- "Investigation pipeline"
- "Status of all investigations"

**Expected**: GROUP BY `INVESTIGATION_STATUS`

---

### Benchmark Set 9: ROI Analysis
- "What is the investigation ROI?"
- "Return on investigation spend"
- "Are investigations profitable?"
- "Cost vs recovery analysis"

**Expected**: `(SUM(RECOVERY_AMOUNT) - SUM(INVESTIGATION_COST)) / SUM(INVESTIGATION_COST) * 100`

---

### Benchmark Set 10: Claims Analysis
- "Claim aging analysis"
- "How old are our open claims?"
- "Claims by aging bucket"
- "Settlement time analysis"

**Expected**: GROUP BY `CLAIM_AGING_BUCKET` from `fact_claim`

---

### Benchmark Set 11: Customer Fraud Patterns
- "Which customers have the most fraud?"
- "Customer fraud analysis"
- "Fraud by customer type"
- "Corporate vs individual fraud"

**Expected**: JOIN to `dim_customer`, GROUP BY customer fields

---

### Benchmark Set 12: AI Insights
- "Show AI insights for investigations"
- "What does AI recommend?"
- "AI priority assessments"
- "AI-generated recommendations"

**Expected**: Query `fraud_ai_results` or `v_fraud_investigation_detail` for `AI_INSIGHTS`, `AI_PRIORITY`, `AI_RECOMMENDATIONS`

---

### Benchmark Set 13: Investigation Duration
- "Average investigation duration"
- "How long do investigations take?"
- "Investigation time by status"
- "Fastest and slowest investigations"

**Expected**: `AVG(INVESTIGATION_DAYS)`, possibly grouped by `INVESTIGATION_STATUS`

---

### Benchmark Set 14: Comprehensive View Queries
- "Show me the full fraud investigation detail"
- "Give me a comprehensive view of investigations"
- "All investigation data with customer and claim info"

**Expected**: Query `v_fraud_investigation_detail` view

---

### Benchmark Set 15: Policy-Level Analysis
- "Fraud by policy type"
- "Which policies are targeted by fraud?"
- "Business line and policy analysis"

**Expected**: JOIN to `dim_policy`, GROUP BY policy fields

## Manual Setup: Joins & Dimensions (UI Only)

The Genie API's `join_specs` proto format is currently undocumented and all standard formats are rejected. Similarly, the `sql_snippets` proto does not support a `dimensions` field. **Add these manually in the Genie Space UI.**

---

### Joins (Configure → Instructions → Joins → + Add)

Add each of these 4 joins:

| # | Left Table | Right Table | Condition |
|---|---|---|---|
| 1 | `fact_fraud_investigation` | `fact_claim` | `fact_fraud_investigation.CLAIM_KEY = fact_claim.CLAIM_KEY` |
| 2 | `fact_claim` | `dim_customer` | `fact_claim.CUSTOMER_KEY = dim_customer.CUSTOMER_KEY AND dim_customer.IS_CURRENT = true` |
| 3 | `fact_claim` | `dim_policy` | `fact_claim.POLICY_KEY = dim_policy.POLICY_KEY` |
| 4 | `fact_fraud_investigation` | `fraud_ai_results` | `fact_fraud_investigation.INVESTIGATION_ID = fraud_ai_results.INVESTIGATION_ID` |

---

### Dimensions (Configure → Instructions → SQL Expressions → + Add → Dimension)

Add each of these 2 dimensions:

| Name | Code | Synonyms |
|---|---|---|
| Investigation Month | `DATE_TRUNC('month', fact_fraud_investigation.FR_DATE)` | month, monthly, by month, time period |
| Risk Tier | `CASE WHEN fact_fraud_investigation.FRAUD_SCORE > 75 THEN 'High' WHEN fact_fraud_investigation.FRAUD_SCORE > 50 THEN 'Medium' ELSE 'Low' END` | risk level, risk category, risk band, risk group |